In [3]:
from transformers import T5Tokenizer, T5Model, TrainingArguments, Trainer, T5ForConditionalGeneration
import pandas as pd 
import numpy as np

In [4]:
from datasets import load_dataset
dataset = load_dataset("knkarthick/samsum")

In [5]:
train_data = dataset["train"].to_pandas()
validation_data = dataset["validation"].to_pandas()
test_data = dataset["test"].to_pandas()

In [6]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\nJ...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\nKim: Bad mood tbh, I was ...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\nSam: i...,"Sam is confused, because he overheard Rick com..."


In [7]:
train_data.shape
validation_data.shape

(818, 3)

In [8]:
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
validation_data = validation_data.sample(n=500, random_state=42).reset_index(drop=True)

DATA PRE-PROCESSING


In [9]:
import re

In [10]:
def clean_data(text):
    text = re.sub(r"\r\n"," ",text) #remove lines
    text = re.sub(r"\s+", " ", text) # remove spaces
    text = re.sub(r"<.*?>"," ", text) #remove html tags
    text = text.strip().lower()
    return text

In [11]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

validation_data["dialogue"] = validation_data["dialogue"].apply(clean_data)
validation_data["summary"] = validation_data["summary"].apply(clean_data)

TOKENIZE


In [12]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [13]:
#use this to set the max_length

import numpy as np
lengths = [
    len(tokenizer.encode(text))
    for text in train_data["dialogue"][:1000]
]

print(max(lengths))
print(sum(lengths) / len(lengths))
print(np.percentile(lengths, [50, 75, 90, 95, 99]))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (740 > 512). Running this sequence through the model will result in indexing errors


919
167.614
[134.5  226.5  339.2  414.25 625.05]


In [14]:
#raw data => tokenized input

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    target = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = target["input_ids"] # token ids => add to input as labels
    return inputs #input => input_ids:dialogue token_ids 1<EOS>, attention_masks:0-padding, 1-actual token, labels:correct summary token

In [15]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
validation_dataset = validation_data.apply(tokenize, axis=1).tolist()
from datasets import Dataset

train_dataset = Dataset.from_list(train_dataset)
validation_dataset = Dataset.from_list(validation_dataset)

BUILDING THE MODEL


In [16]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

TRAIN THE MODEL


In [17]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:",device)
model.to(device)

device: cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [18]:
#Training Arguments
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500
)

In [19]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset= train_dataset,
    eval_dataset= validation_dataset
)

In [20]:
#train the model
trainer.train()

c:\Users\sarin\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
#model load => fine-tune => save the model
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [ ]:
#load the model
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

TEST THE MODEL => CORE LOGIC FOR SUMMARIZATION

In [ ]:
def summarize_dialogue(dialogue):
    #clean the data
    dialogue = clean_data(dialogue)
    #tokenize
    inputs = tokenizer(
        dialogue,
        padding = "max_length",
        max_length = 512,
        truncation = True,
        return_tensors = "pt"#return as pytorch tensors
    )
    #generate the summary => token ids
    targets = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams = 4,
        early_stopping = True
    )
    #decode the token ids : convert to summary
    summary = tokenizer.decode(targets[0], skip_special_tokens=True)
    return summary

    

In [ ]:
test_dialogue = """Reporter: Thank you for joining us today. Could you explain why artificial intelligence has become so important in healthcare?

Expert: AI can process large amounts of medical data much faster than humans. It can help doctors identify patterns in medical images, predict potential health risks, and support clinical decision-making.

Reporter: Does that mean AI will replace doctors in the future?

Expert: No. AI is better viewed as a decision-support tool. Doctors provide clinical judgment, understand the patient's circumstances, and make the final decisions. AI can assist them, but it should not replace human expertise.

Reporter: What are some of the biggest challenges in using AI in healthcare?

Expert: Data privacy is a major concern because medical records contain highly sensitive information. Another challenge is bias. If an AI system is trained on limited or unrepresentative data, its predictions may not work equally well for all groups of patients.

Reporter: How can these problems be addressed?

Expert: Healthcare organizations need strong data protection policies, diverse training datasets, regular testing, and human oversight. AI systems should also be evaluated continuously after deployment to make sure they remain accurate and safe.

Reporter: So what do you think the future of AI in healthcare will look like?

Expert: I expect AI to become increasingly integrated into healthcare workflows, helping professionals analyze information and make better-informed decisions while keeping humans responsible for important clinical choices.

Reporter: Thank you for your insights.

Expert: Thank you for having me."""

final_summary = summarize_dialogue(test_dialogue)
print(final_summary)


ai can process large amounts of medical data much faster than humans. it can help doctors identify patterns in medical images, predict potential health risks, and support clinical decision-making. ai is better viewed as a decision-support tool.
